In [82]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q
!pip install yfinance -q

In [83]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [84]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [85]:
import boto3
import os
import glob
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import DataFrame
from functools import reduce

# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Download All Partitioned Files from MinIO ─────────────────────────────────
local_dir  = "/tmp/CGS_Portfolio"
bucket     = "rawdatasets"
prefix     = "CGS_Portfolio/"

os.makedirs(local_dir, exist_ok=True)

response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    s3_key = obj["Key"]

    # Skip if the key is just the folder prefix itself
    if s3_key.endswith("/"):
        continue

    # Reconstruct local path preserving year=/month= structure
    relative_path = os.path.relpath(s3_key, prefix)
    local_file_path = os.path.join(local_dir, relative_path)

    # Create subdirectories if needed (e.g. year=2023/month=6/)
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_file_path)
    #print(f"Downloaded: {s3_key} → {local_file_path}")

# ── Read All Partitions into Spark DataFrame ──────────────────────────────────
spark = SparkSession.builder \
    .appName("ReadCSGCSV") \
    .getOrCreate()

dfs = []

for filename in os.listdir(local_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(local_dir, filename)

        # Get file metadata
        created_time = os.path.getctime(filepath)
        date_created = datetime.fromtimestamp(created_time).strftime("%Y-%m-%d")

        # Read each CSV and append metadata columns
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(filepath)

        df = df.withColumn("source_file", F.lit(filename)) \
               .withColumn("date_created", F.lit(date_created))

        dfs.append(df)

# Union all DataFrames into one
final_df = reduce(DataFrame.union, dfs)

#final_df.printSchema()
#final_df.show()

In [86]:
import re

def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # Remove special characters, keep only alphanumeric and underscores
        cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', col)
        # Remove leading/trailing underscores
        cleaned = cleaned.strip('_')
        # Replace multiple consecutive underscores with single one
        cleaned = re.sub(r'_+', '_', cleaned)
        new_columns.append(cleaned)

    # Rename all columns
    for old, new in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old, new)

    return df

# Apply to your DataFrame
final_df = clean_column_names(final_df)
#final_df.printSchema()
#final_df.show()

In [87]:
import yfinance as yf
import pandas as pd

# Decode bytes → string → wrap in StringIO for pandas
pdf = final_df.toPandas()

def get_info(symbol):
    try:
        info = yf.Ticker(symbol).info
        return pd.Series({
            "name": info.get("longName", "N/A"),
            "current_price": info.get("currentPrice", info.get("regularMarketPrice", None)),
            "shortName": info.get("shortName", "N/A"),
            "sector": info.get("sector", "N/A"),
            "industry": info.get("industry", "N/A"),
            "country": info.get("country", "N/A"),
            "city": info.get("city", "N/A"),
            "state": info.get("state", "N/A"),
            "website": info.get("website", "N/A"),
            "marketCap": round(float(info.get("marketCap")) / 1_000_000, 2) if info.get("marketCap") else None,
            "enterpriseToRevenue": info.get("enterpriseToRevenue", None),
            "dividendYield": info.get("dividendYield", None),
            "dividendRate": info.get("dividendRate", None),
            "payoutRatio": info.get("payoutRatio", None),
            "exDividendDate": pd.to_datetime(info.get("exDividendDate"), unit="s").date() if info.get("exDividendDate") else None,
            "lastDividendDate": pd.to_datetime(info.get("lastDividendDate"), unit="s").date() if info.get("lastDividendDate") else None,
            "lastDividendValue": info.get("lastDividendValue", None),
            "lastSplitFactor": info.get("lastSplitFactor", None),
            "lastSplitDate": pd.to_datetime(info.get("lastSplitDate"), unit="s").date() if info.get("lastSplitDate") else None,
        })
    except:
        return pd.Series({"name": "N/A", "current_price": None})

# Apply row-wise — adds 2 new columns directly
pdf[["name", "current_price", "shortName", "sector", "industry", "country"
, "city", "state", "website", "marketCap", "enterpriseToRevenue", "dividendYield"
, "dividendRate", "payoutRatio" , "exDividendDate", "lastDividendDate", "lastDividendValue"
, "lastSplitFactor", "lastSplitDate"
]] = pdf["Code"].apply(get_info)

df = spark.createDataFrame(pdf)

df.printSchema()
df.show()

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AIMA.SI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AMOE.SI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ESRE.SI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LIHS.SI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: STON.SI"}}}


root
 |-- Symbol: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- LACP: double (nullable = true)
 |-- Qty_Hand: long (nullable = true)
 |-- Qty_Avai: long (nullable = true)
 |-- Qty_Q_S: long (nullable = true)
 |-- Avg_Buy_Prc: double (nullable = true)
 |-- Last: double (nullable = true)
 |-- Mkt_Val: double (nullable = true)
 |-- Un_G_L: double (nullable = true)
 |-- Chg: string (nullable = true)
 |-- Chg_Perc: string (nullable = true)
 |-- Yr_High: double (nullable = true)
 |-- Yr_Low: double (nullable = true)
 |-- Day_High: double (nullable = true)
 |-- Day_Low: double (nullable = true)
 |-- Vol: long (nullable = true)
 |-- Lot_Size: long (nullable = true)
 |-- Un_G_L_Perc: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Exchg: string (nullable = true)
 |-- Qty_Sold: long (nullable = true)
 |-- Qty_Susp: long (nullable = true)
 |-- Bid: double (nullable = true)
 |-- Ask: double (nullable = true)
 |-- Avg_Sell_Price: double (nullable = true)
 |

In [88]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "cgs_portfolio_yahoo"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [89]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "Symbol",
    "Code",
    "LACP",
    "Qty_Hand",
    "Qty_Avai",
    "Qty_Q_S",
    "Avg_Buy_Prc",
    "Last",
    "Mkt_Val",
    "Un_G_L",
    "Chg",
    "Chg_Perc",
    "Yr_High",
    "Yr_Low",
    "Day_High",
    "Day_Low",
    "Vol",
    "Lot_Size",
    "Un_G_L_Perc",
    "Currency",
    "Exchg",
    "Qty_Sold",
    "Qty_Susp",
    "Bid",
    "Ask",
    "Avg_Sell_Price",
    "source_file",
    "date_created",
    "name",
    "current_price",
    "shortName",
    "sector",
    "industry",
    "country",
    "city",
    "state",
    "website",
    "marketCap",
    "enterpriseToRevenue",
    "dividendYield",
    "dividendRate",
    "payoutRatio",
    "exDividendDate",
    "lastDividendDate",
    "lastDividendValue",
    "lastSplitFactor",
    "lastSplitDate",
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
              Symbol                STRING,
              Code                  STRING,
              LACP                  DOUBLE,
              Qty_Hand              BIGINT,
              Qty_Avai              BIGINT,
              Qty_Q_S               BIGINT,
              Avg_Buy_Prc           DOUBLE,
              Last                  DOUBLE,
              Mkt_Val               DOUBLE,
              Un_G_L                DOUBLE,
              Chg                   STRING,
              Chg_Perc              STRING,
              Yr_High               DOUBLE,
              Yr_Low                DOUBLE,
              Day_High              DOUBLE,
              Day_Low               DOUBLE,
              Vol                   BIGINT,
              Lot_Size              BIGINT,
              Un_G_L_Perc           DOUBLE,
              Currency              STRING,
              Exchg                 STRING,
              Qty_Sold              BIGINT,
              Qty_Susp              BIGINT,
              Bid                   DOUBLE,
              Ask                   DOUBLE,
              Avg_Sell_Price        DOUBLE,
              source_file           STRING,
              date_created          STRING,
              name                  STRING,
              current_price         DOUBLE,
              shortName             STRING,
              sector                STRING,
              industry              STRING,
              country               STRING,
              city                  STRING,
              state                 STRING,
              website               STRING,
              marketCap             DOUBLE,
              enterpriseToRevenue   DOUBLE,
              dividendYield         DOUBLE,
              dividendRate          DOUBLE,
              payoutRatio           DOUBLE,
              exDividendDate        DATE,
              lastDividendDate      DATE,
              lastDividendValue     DOUBLE,
              lastSplitFactor       STRING,
              lastSplitDate         DATE
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────
        #for _, row in pdf.iterrows():
        #    insert_sql = f"""
        #    INSERT INTO {FULL_TABLE} (id, name, score, is_active)
        #    VALUES ({row['id']}, '{row['name']}', {row['score']}, {str(row['is_active']).upper()})
        #    """
        #    cursor.execute(insert_sql)

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df.collect()]

        #cursor.executemany(
        #    f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        #    rows
        #)

        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE Symbol = ?
              AND Code = ?
              AND source_file = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("Symbol")],
                  row[COLUMNS.index("Code")],
                  row[COLUMNS.index("source_file")])
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.cgs_portfolio_yahoo' created/verified.
✅ Inserted 98 rows into 'workspace.default.cgs_portfolio_yahoo'.

📦 Data in 'workspace.default.cgs_portfolio_yahoo':
Row(Symbol='PANAMY.KL', Code='3719.KL', LACP=7.269999980926514, Qty_Hand=100, Qty_Avai=100, Qty_Q_S=0, Avg_Buy_Prc=25.200000762939453, Last=7.230000019073486, Mkt_Val=723.0, Un_G_L=-1797.0, Chg='-0.040', Chg_Perc='-0.55', Yr_High=15.324999809265137, Yr_Low=6.785999774932861, Day_High=7.409999847412109, Day_Low=7.210000038146973, Vol=51100, Lot_Size=100, Un_G_L_Perc=-71.30999755859375, Currency='MYR', Exchg='KL', Qty_Sold=0, Qty_Susp=0, Bid=7.21999979019165, Ask=7.239999771118164, Avg_Sell_Price=0.0, source_file='20260303.csv', date_created='2026-03-04', name='Panasonic Manufacturing Malaysia Berhad', current_price=7.050000190734863, shortName='PANAMY', sector='Consumer Cyclical', industry='Furnishings, Fixtures & Appliances', country='Malaysia', city='Shah Alam', state='N/A', website='https://pmma.my.panas

In [90]:
spark.stop()